# 2026 COMP90042 Project — Classifier Branch v5

基于 v3。检索固定为 BM25 top500 + CE reranker + dynamic selector；分类器固定为 joint MiniLM 4-way，不换模型、不换架构。一次运行完成 input top-k、max length、gold→retrieved curriculum、soft class weights 的控制变量实验。

In [1]:
# Optional dependency installation. Safe in Colab; usually skipped locally if packages already exist.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "bm25s": "bm25s",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "psutil": "psutil",
}

for pip_name, import_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# 1. Data and config

In [2]:

from pathlib import Path
import json
import random
import time
import pickle
import re
import gc
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# -------------------------
# Classifier branch config
# -------------------------
SEED = 42
FAST_DEV_MODE = False  # True = quick smoke test; False = full project run

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs_notebook_classifier_branch")
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
CACHE_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip("-")


# Fixed retrieval baseline from v3. Do not repeat top1000 or stance experiments here.
BM25_CANDIDATE_K = 500 if not FAST_DEV_MODE else 50
FIXED_K_GRID = [2, 3, 4, 5]
THRESHOLD_GRID = [round(float(x), 2) for x in np.arange(0.02, 0.52, 0.02)] + [0.60, 0.70, 0.80]
RELATIVE_LOGIT_DELTA_GRID = [0.50, 0.75, 1.00, 1.50, 2.00, 3.00]
MIN_FINAL_K = 1
MAX_FINAL_K = 5
FINAL_RETRIEVAL_POLICY = "prefer_dynamic"
DYNAMIC_RETRIEVAL_TOLERANCE = 0.02

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_MAX_LEN = 256
RERANKER_BATCH_SIZE = 32 if not FAST_DEV_MODE else 8
RERANKER_EVAL_BATCH_SIZE = 128 if not FAST_DEV_MODE else 16
RERANKER_EPOCHS = 3 if not FAST_DEV_MODE else 1
RERANKER_LR = 1e-5
NEGATIVES_PER_POSITIVE = 4 if not FAST_DEV_MODE else 2
HARD_NEGATIVE_POOL = min(200, BM25_CANDIDATE_K)

# Joint classifier fixed model family. Branch only changes input/top-k/max-len/curriculum/soft weights.
LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
CLASSIFIER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TRAIN_BATCH_SIZE = 16 if not FAST_DEV_MODE else 4
EVAL_BATCH_SIZE = 32 if not FAST_DEV_MODE else 8
CLASSIFIER_LR = 1e-5
WEIGHT_DECAY = 0.01
FALLBACK_TO_MAJORITY_IF_CLASSIFIER_WORSE = True
CLASSIFIER_MIN_H_IMPROVEMENT = 0.005

# Cache/checkpoint switches.
SAVE_BM25_INDEX = True
SAVE_MODEL_CHECKPOINTS = True
FORCE_REBUILD_BM25_INDEX = False
FORCE_RECOMPUTE_BM25_CANDIDATES = False
FORCE_RERANKER_RETRAIN = False
FORCE_RESCORE_CE = False
FORCE_CLASSIFIER_RETRAIN = False

# Controlled classifier experiment queue.
# class_weight_power: 0.0 = no weights; 0.25/0.5 = soft inverse-frequency weights; raw 1.0 intentionally excluded.
# train_schedule: [(source, epochs)], source in {gold, retrieved, gold_plus_retrieved}.
CLASSIFIER_EXPERIMENTS = [
    {"name": "v3_baseline_top5_len256_gpr_w0", "input_top_k": 5, "max_seq_len": 256, "class_weight_power": 0.0, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top2_len384_gpr_w0", "input_top_k": 2, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top3_len384_gpr_w0", "input_top_k": 3, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top4_len384_gpr_w0", "input_top_k": 4, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top5_len384_gpr_w0", "input_top_k": 5, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top3_len384_gpr_w025", "input_top_k": 3, "max_seq_len": 384, "class_weight_power": 0.25, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top4_len384_gpr_w025", "input_top_k": 4, "max_seq_len": 384, "class_weight_power": 0.25, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top3_len384_gpr_w05", "input_top_k": 3, "max_seq_len": 384, "class_weight_power": 0.50, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top4_len384_gpr_w05", "input_top_k": 4, "max_seq_len": 384, "class_weight_power": 0.50, "train_schedule": [("gold_plus_retrieved", 8)]},
    {"name": "top3_len384_gold2_retrieved6_w0", "input_top_k": 3, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold", 2), ("retrieved", 6)]},
    {"name": "top4_len384_gold2_retrieved6_w0", "input_top_k": 4, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold", 2), ("retrieved", 6)]},
    {"name": "top3_len384_gold2_gpr6_w0", "input_top_k": 3, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold", 2), ("gold_plus_retrieved", 6)]},
    {"name": "top4_len384_gold2_gpr6_w0", "input_top_k": 4, "max_seq_len": 384, "class_weight_power": 0.0, "train_schedule": [("gold", 2), ("gold_plus_retrieved", 6)]},
]
if FAST_DEV_MODE:
    CLASSIFIER_EXPERIMENTS = CLASSIFIER_EXPERIMENTS[:2]
    for e in CLASSIFIER_EXPERIMENTS:
        e["train_schedule"] = [(e["train_schedule"][0][0], 1)]

CACHE_VERSION = "classifier_branch_v5_from_v3_top500_joint"
CACHE_TAG = f"{CACHE_VERSION}_{safe_name(RERANKER_MODEL_NAME)}_top{BM25_CANDIDATE_K}_seed{SEED}"


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(SEED)


Device: cuda


D:\_Search\_Study\COMP90042-NLP\A3_Group\COMP90042_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


train_claims = load_json(DATA_DIR / "train-claims.json")
dev_claims = load_json(DATA_DIR / "dev-claims.json")
test_claims = load_json(DATA_DIR / "test-claims-unlabelled.json")
evidence = load_json(DATA_DIR / "evidence.json")

if FAST_DEV_MODE:
    train_claims = dict(list(train_claims.items())[:80])
    dev_claims = dict(list(dev_claims.items())[:30])
    test_claims = dict(list(test_claims.items())[:30])

print(f"train:    {len(train_claims)}")
print(f"dev:      {len(dev_claims)}")
print(f"test:     {len(test_claims)}")
print(f"evidence: {len(evidence)}")

train:    1228
dev:      154
test:     153
evidence: 1208827


In [4]:
# Lightweight EDA used to justify later choices.
label_dist = Counter(c["claim_label"] for c in train_claims.values())
print("Train label distribution:")
for lbl in LABELS:
    cnt = label_dist[lbl]
    print(f"  {lbl:20s} {cnt:5d} ({cnt / len(train_claims):6.2%})")

gt_counts = [len(c["evidences"]) for c in train_claims.values()]
print("\nGround-truth evidence count per train claim:")
print("  min=", min(gt_counts), "max=", max(gt_counts), "mean=", round(float(np.mean(gt_counts)), 3))
print("  exact counts:", sorted(Counter(gt_counts).items()))

Train label distribution:
  SUPPORTS               519 (42.26%)
  REFUTES                199 (16.21%)
  NOT_ENOUGH_INFO        386 (31.43%)
  DISPUTED               124 (10.10%)

Ground-truth evidence count per train claim:
  min= 1 max= 5 mean= 3.357
  exact counts: [(1, 210), (2, 223), (3, 191), (4, 127), (5, 477)]


In [5]:
# Official-style metrics. These mirror eval.py's logic and let us tune inside the notebook.
def evidence_f1_for_claim(pred_eids, gold_eids):
    pred_eids = list(pred_eids)
    gold_eids = list(gold_eids)
    if len(pred_eids) == 0:
        return 0.0
    pred_set = set(pred_eids)
    correct = sum(1 for eid in gold_eids if eid in pred_set)
    if correct == 0:
        return 0.0
    precision = correct / len(pred_eids)
    recall = correct / len(gold_eids)
    return 2 * precision * recall / (precision + recall)


def evaluate_submission(predictions, gold_claims, verbose=True):
    f_scores = []
    correct_labels = 0
    total = 0
    for cid, gold in gold_claims.items():
        pred = predictions[cid]
        f_scores.append(evidence_f1_for_claim(pred["evidences"], gold["evidences"]))
        correct_labels += int(pred["claim_label"] == gold["claim_label"])
        total += 1
    F = float(np.mean(f_scores))
    A = correct_labels / total
    H = 0.0 if (F + A) == 0 else 2 * F * A / (F + A)
    if verbose:
        print(f"Evidence Retrieval F-score (F)    = {F:.6f}")
        print(f"Claim Classification Accuracy (A) = {A:.6f}")
        print(f"Harmonic Mean of F and A          = {H:.6f}")
    return {"F": F, "A": A, "H": H}


def evaluate_retrieval_only(retrieval, gold_claims):
    return float(
        np.mean(
            [
                evidence_f1_for_claim(retrieval[cid], claim["evidences"])
                for cid, claim in gold_claims.items()
            ]
        )
    )


def majority_label(claims):
    return Counter(c["claim_label"] for c in claims.values()).most_common(1)[0][0]


def validate_retrieval_coverage(claims_dict, retrieval, split_name="split"):
    """Fail early with a helpful message if retrieval is incomplete or empty."""
    missing_cids = [cid for cid in claims_dict if cid not in retrieval]
    if missing_cids:
        raise KeyError(f"{split_name}: retrieval missing {len(missing_cids)} claim ids, e.g. {missing_cids[:3]}")

    empty_cids = [cid for cid in claims_dict if len(retrieval[cid]) == 0]
    if empty_cids:
        raise ValueError(f"{split_name}: retrieval has empty evidence lists, e.g. {empty_cids[:3]}")

    bad_eids = []
    for cid in claims_dict:
        for eid in retrieval[cid]:
            if eid not in evidence:
                bad_eids.append((cid, eid))
                if len(bad_eids) >= 3:
                    break
        if len(bad_eids) >= 3:
            break
    if bad_eids:
        raise KeyError(f"{split_name}: retrieval contains unknown evidence ids, e.g. {bad_eids}")


def build_predictions(claims, retrieval, label_predictions=None, default_label=None):
    if label_predictions is None:
        assert default_label is not None
        label_predictions = {cid: default_label for cid in claims.keys()}
    out = {}
    fallback_eid = next(iter(evidence.keys()))
    for cid in claims.keys():
        eids = list(retrieval.get(cid, []))
        if len(eids) == 0:
            # Assignment requires at least one evidence. This fallback should rarely trigger.
            eids = [fallback_eid]
        out[cid] = {"claim_label": label_predictions[cid], "evidences": eids}
    return out


def write_predictions(predictions, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # ensure_ascii=True keeps the file readable even when Windows uses a non-UTF-8 default encoding.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=True)
    print("Wrote", path)


## Fixed v3 retrieval pipeline

In [6]:

import bm25s
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# -------------------------
# Fixed v3 retrieval pipeline with caching
# -------------------------
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

BM25_SPEC = {"name": "baseline_stop_en", "stopwords": "en", "k1": 1.5, "b": 0.75}


def bm25_spec_name(spec=BM25_SPEC):
    stop = "nostop" if spec.get("stopwords") is None else f"stop-{spec.get('stopwords')}"
    return safe_name(f"{spec['name']}_{stop}_k1{spec.get('k1', 1.5)}_b{spec.get('b', 0.75)}")


def build_or_load_bm25_index():
    name = bm25_spec_name()
    index_dir = CACHE_DIR / f"bm25s_index_{name}_{len(evidence_ids)}docs"
    retriever = None
    if SAVE_BM25_INDEX and index_dir.exists() and not FORCE_REBUILD_BM25_INDEX:
        try:
            print("Loading BM25 index:", index_dir)
            retriever = bm25s.BM25.load(str(index_dir), load_corpus=False)
        except Exception as exc:
            print("Could not load BM25 index; rebuilding. Reason:", repr(exc))
            retriever = None
    if retriever is None:
        print(f"Tokenizing/building BM25 index: {name}")
        corpus_tokens = bm25s.tokenize(evidence_texts, stopwords="en", stemmer=None)
        retriever = bm25s.BM25(k1=1.5, b=0.75)
        retriever.index(corpus_tokens)
        if SAVE_BM25_INDEX:
            try:
                retriever.save(str(index_dir))
            except Exception as exc:
                print("Warning: BM25 save failed:", repr(exc))
    return retriever


bm25_retriever = build_or_load_bm25_index()


def bm25_retrieve_with_scores(claims_dict, k=BM25_CANDIDATE_K):
    cids = list(claims_dict.keys())
    queries = [claims_dict[cid]["claim_text"] for cid in cids]
    query_tokens = bm25s.tokenize(queries, stopwords="en", stemmer=None)
    results, scores = bm25_retriever.retrieve(query_tokens, k=k)
    out = {}
    for i, cid in enumerate(cids):
        out[cid] = [(evidence_ids[int(j)], float(s)) for j, s in zip(results[i], scores[i])]
    return out


def compute_or_load_bm25_candidates(claims_dict, split_name, k=BM25_CANDIDATE_K):
    cache_file = CACHE_DIR / f"{split_name}_bm25_{bm25_spec_name()}_top{k}.pkl"
    if cache_file.exists() and not FORCE_RECOMPUTE_BM25_CANDIDATES:
        print("Loading cached BM25 candidates:", cache_file)
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    print(f"Computing BM25 candidates for {split_name} ...")
    candidates = bm25_retrieve_with_scores(claims_dict, k=k)
    with open(cache_file, "wb") as f:
        pickle.dump(candidates, f)
    return candidates


def strip_scores(candidate_cache, k):
    return {cid: [eid for eid, _ in pairs[:k]] for cid, pairs in candidate_cache.items()}


class RerankerPairDataset(Dataset):
    def __init__(self, examples, claims_dict, evidence_dict, tokenizer, max_len=RERANKER_MAX_LEN):
        self.examples = examples
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(
            self.claims[ex["cid"]]["claim_text"],
            self.evidence[ex["eid"]],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(float(ex["label"]), dtype=torch.float32)
        return item


def build_reranker_examples(claims_dict, bm25_candidates, negatives_per_positive=NEGATIVES_PER_POSITIVE):
    examples = []
    rng = random.Random(SEED)
    for cid, claim in claims_dict.items():
        gold = [eid for eid in claim["evidences"] if eid in evidence]
        gold_set = set(gold)
        for eid in gold:
            examples.append({"cid": cid, "eid": eid, "label": 1.0})
        candidate_negs = []
        seen = set()
        for eid, _ in bm25_candidates[cid][:HARD_NEGATIVE_POOL]:
            if eid in gold_set or eid in seen or eid not in evidence:
                continue
            candidate_negs.append(eid)
            seen.add(eid)
        needed = negatives_per_positive * max(1, len(gold))
        if len(candidate_negs) > needed:
            candidate_negs = rng.sample(candidate_negs, needed)
        for eid in candidate_negs:
            examples.append({"cid": cid, "eid": eid, "label": 0.0})
    rng.shuffle(examples)
    return examples


reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)


def load_fresh_reranker():
    return AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_NAME,
        num_labels=1,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)


@torch.no_grad()
def score_candidates_with_reranker(model, claims_dict, bm25_candidates, split_name="dev", allow_cache=True):
    score_cache_file = CACHE_DIR / f"{split_name}_{CACHE_TAG}_ce_scores.pkl"
    if allow_cache and score_cache_file.exists() and not FORCE_RESCORE_CE:
        print("Loading cached CE scores:", score_cache_file)
        with open(score_cache_file, "rb") as f:
            return pickle.load(f)
    model.eval()
    score_cache = {}
    print(f"Scoring {split_name} candidates with cross-encoder reranker...")
    for cid, claim in tqdm(list(claims_dict.items())):
        cand_pairs = bm25_candidates[cid]
        cand_eids = [eid for eid, _ in cand_pairs]
        bm25_scores = np.array([score for _, score in cand_pairs], dtype=np.float32)
        logits_all = []
        for start in range(0, len(cand_eids), RERANKER_EVAL_BATCH_SIZE):
            batch_eids = cand_eids[start:start + RERANKER_EVAL_BATCH_SIZE]
            enc = reranker_tokenizer(
                [claim["claim_text"]] * len(batch_eids),
                [evidence[eid] for eid in batch_eids],
                truncation=True,
                padding=True,
                max_length=RERANKER_MAX_LEN,
                return_tensors="pt",
            ).to(DEVICE)
            logits = model(**enc).logits.squeeze(-1)
            logits_all.extend(logits.detach().cpu().tolist())
        logits_np = np.array(logits_all, dtype=np.float32)
        score_cache[cid] = {"eids": cand_eids, "bm25": bm25_scores, "ce_logit": logits_np, "ce_prob": 1.0 / (1.0 + np.exp(-logits_np))}
    with open(score_cache_file, "wb") as f:
        pickle.dump(score_cache, f)
    print("Cached CE scores:", score_cache_file)
    return score_cache


def select_fixed_k_ce(score_cache, k):
    return {cid: [entry["eids"][int(i)] for i in np.argsort(-entry["ce_logit"])[:k]] for cid, entry in score_cache.items()}


def select_dynamic_threshold_ce(score_cache, threshold, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K):
    out = {}
    for cid, entry in score_cache.items():
        probs = entry["ce_prob"]
        order = np.argsort(-probs)
        selected = [int(i) for i in order[:max_k] if probs[int(i)] >= threshold]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def select_relative_logit_ce(score_cache, delta, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K):
    out = {}
    for cid, entry in score_cache.items():
        logits = entry["ce_logit"]
        order = np.argsort(-logits)
        top = float(logits[int(order[0])])
        selected = [int(i) for i in order[:max_k] if top - float(logits[int(i)]) <= delta]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def tune_retrieval_from_ce_scores(dev_score_cache):
    rows = []
    for k in FIXED_K_GRID:
        retr = select_fixed_k_ce(dev_score_cache, k)
        rows.append({"mode": "fixed_k", "k": k, "threshold": np.nan, "delta": np.nan, "retrieval_F": evaluate_retrieval_only(retr, dev_claims), "avg_pred_evidence": np.mean([len(v) for v in retr.values()])})
    for threshold in THRESHOLD_GRID:
        retr = select_dynamic_threshold_ce(dev_score_cache, threshold)
        rows.append({"mode": "dynamic_threshold", "k": np.nan, "threshold": threshold, "delta": np.nan, "retrieval_F": evaluate_retrieval_only(retr, dev_claims), "avg_pred_evidence": np.mean([len(v) for v in retr.values()])})
    for delta in RELATIVE_LOGIT_DELTA_GRID:
        retr = select_relative_logit_ce(dev_score_cache, delta)
        rows.append({"mode": "relative_logit", "k": np.nan, "threshold": np.nan, "delta": delta, "retrieval_F": evaluate_retrieval_only(retr, dev_claims), "avg_pred_evidence": np.mean([len(v) for v in retr.values()])})
    return pd.DataFrame(rows).sort_values("retrieval_F", ascending=False).reset_index(drop=True)


def choose_final_retrieval_setting(results_df):
    results_df = results_df.sort_values("retrieval_F", ascending=False).reset_index(drop=True)
    best = results_df.iloc[0]
    if FINAL_RETRIEVAL_POLICY == "best_dev":
        chosen = best
    elif FINAL_RETRIEVAL_POLICY in {"fixed_k", "dynamic_threshold", "relative_logit"}:
        chosen = results_df[results_df["mode"] == FINAL_RETRIEVAL_POLICY].iloc[0]
    elif FINAL_RETRIEVAL_POLICY == "prefer_dynamic":
        dynamic = results_df[results_df["mode"].isin(["dynamic_threshold", "relative_logit"])]
        eligible = dynamic[dynamic["retrieval_F"] >= float(best["retrieval_F"]) - DYNAMIC_RETRIEVAL_TOLERANCE]
        chosen = eligible.iloc[0] if not eligible.empty else best
    else:
        raise ValueError(FINAL_RETRIEVAL_POLICY)
    print("Best dev row:", best.to_dict())
    print("Chosen row:", chosen.to_dict())
    return chosen.to_dict()


def apply_retrieval_setting(score_cache, row):
    if row["mode"] == "fixed_k":
        return select_fixed_k_ce(score_cache, int(row["k"]))
    if row["mode"] == "dynamic_threshold":
        return select_dynamic_threshold_ce(score_cache, float(row["threshold"]))
    if row["mode"] == "relative_logit":
        return select_relative_logit_ce(score_cache, float(row["delta"]))
    raise ValueError(row)


Tokenizing/building BM25 index: baseline_stop_en_stop-en_k11.5_b0.75


In [7]:

# Build/load fixed retrieval from v3. Classifier experiments all share this retrieval.
train_bm25_candidates = compute_or_load_bm25_candidates(train_claims, "train", BM25_CANDIDATE_K)
dev_bm25_candidates = compute_or_load_bm25_candidates(dev_claims, "dev", BM25_CANDIDATE_K)

reranker_ckpt_path = OUTPUT_DIR / f"reranker_best_{CACHE_TAG}.pt"
reranker_model = load_fresh_reranker()

freshly_trained_reranker = False
if SAVE_MODEL_CHECKPOINTS and reranker_ckpt_path.exists() and not FORCE_RERANKER_RETRAIN:
    print("Loading cached reranker checkpoint:", reranker_ckpt_path)
    ckpt = torch.load(reranker_ckpt_path, map_location=DEVICE)
    reranker_model.load_state_dict(ckpt["model_state"])
    best_reranker_row = ckpt.get("best_row")
else:
    freshly_trained_reranker = True
    reranker_train_examples = build_reranker_examples(train_claims, train_bm25_candidates)
    pos = sum(1 for x in reranker_train_examples if x["label"] == 1.0)
    neg = len(reranker_train_examples) - pos
    print(f"Reranker examples: {len(reranker_train_examples):,} | positives={pos:,} negatives={neg:,}")
    reranker_ds = RerankerPairDataset(reranker_train_examples, train_claims, evidence, reranker_tokenizer)
    reranker_loader = DataLoader(reranker_ds, batch_size=RERANKER_BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(reranker_model.parameters(), lr=RERANKER_LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(reranker_loader) * RERANKER_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_state = None
    best_reranker_row = None
    best_F = -1.0
    for epoch in range(1, RERANKER_EPOCHS + 1):
        reranker_model.train()
        losses = []
        t0 = time.time()
        for batch in tqdm(reranker_loader, desc=f"Reranker epoch {epoch}/{RERANKER_EPOCHS}"):
            labels = batch.pop("labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = reranker_model(**batch).logits.squeeze(-1)
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reranker_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss.item()))
        tmp_scores = score_candidates_with_reranker(reranker_model, dev_claims, dev_bm25_candidates, f"dev_epoch{epoch}", allow_cache=False)
        tmp_df = tune_retrieval_from_ce_scores(tmp_scores)
        row = choose_final_retrieval_setting(tmp_df)
        print(f"Epoch {epoch}: loss={np.mean(losses):.4f} | retrieval_F={row['retrieval_F']:.4f} | time={time.time()-t0:.1f}s")
        if row["retrieval_F"] > best_F:
            best_F = row["retrieval_F"]
            best_reranker_row = row
            best_state = {k: v.detach().cpu().clone() for k, v in reranker_model.state_dict().items()}
    if best_state is not None:
        reranker_model.load_state_dict(best_state)
    if SAVE_MODEL_CHECKPOINTS:
        torch.save({"model_state": reranker_model.state_dict(), "best_row": best_reranker_row}, reranker_ckpt_path)
        print("Saved reranker checkpoint:", reranker_ckpt_path)

# Final CE scores and retrieval.
dev_ce_scores = score_candidates_with_reranker(reranker_model, dev_claims, dev_bm25_candidates, "dev_final", allow_cache=not freshly_trained_reranker)
retrieval_results = tune_retrieval_from_ce_scores(dev_ce_scores)
display(retrieval_results.head(30))
best_row = choose_final_retrieval_setting(retrieval_results)
dev_retrieval = apply_retrieval_setting(dev_ce_scores, best_row)
validate_retrieval_coverage(dev_claims, dev_retrieval, split_name="dev")

train_ce_scores = score_candidates_with_reranker(reranker_model, train_claims, train_bm25_candidates, "train_final", allow_cache=not freshly_trained_reranker)
train_retrieval = apply_retrieval_setting(train_ce_scores, best_row)
validate_retrieval_coverage(train_claims, train_retrieval, split_name="train")

majority = majority_label(train_claims)
retrieval_majority_predictions = build_predictions(dev_claims, dev_retrieval, default_label=majority)
print("Dev score with fixed retrieval + majority label:")
retrieval_majority_metrics = evaluate_submission(retrieval_majority_predictions, dev_claims)


Computing BM25 candidates for train ...


Computing BM25 candidates for dev ...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11302.79it/s]


Reranker examples: 20,610 | positives=4,122 negatives=16,488


Reranker epoch 1/3: 100%|██████████| 645/645 [01:47<00:00,  5.99it/s]


...

Cached CE scores: outputs_notebook_classifier_branch\cache\dev_final_classifier_branch_v5_from_v3_top500_joint_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_ce_scores.pkl


,mode,k,threshold,delta,retrieval_F,avg_pred_evidence
0,relative_logit,NaN,NaN,1.50,0.210560,4.246753
1,relative_logit,NaN,NaN,2.00,0.204216,4.623377
2,relative_logit,NaN,NaN,1.00,0.194712,3.584416
3,fixed_k,4.0,NaN,NaN,0.193434,4.000000
4,relative_logit,NaN,NaN,3.00,0.189775,4.863636
5,dynamic_threshold,NaN,0.42,NaN,0.186467,4.532468
6,dynamic_threshold,NaN,0.40,NaN,0.185622,4.590909
7,dynamic_threshold,NaN,0.38,NaN,0.185390,4.610390
8,fixed_k,3.0,NaN,NaN,0.185297,3.000000
9,dynamic_threshold,NaN,0.44,NaN,0.184395,4.467532


Best dev row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 1.5, 'retrieval_F': 0.21055967841682127, 'avg_pred_evidence': 4.246753246753247}
Chosen row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 1.5, 'retrieval_F': 0.21055967841682127, 'avg_pred_evidence': 4.246753246753247}
Scoring train_final candidates with cross-encoder reranker...


100%|██████████| 1228/1228 [06:23<00:00,  3.20it/s]

Cached CE scores: outputs_notebook_classifier_branch\cache\train_final_classifier_branch_v5_from_v3_top500_joint_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_ce_scores.pkl
Dev score with fixed retrieval + majority label:
Evidence Retrieval F-score (F)    = 0.210560
Claim Classification Accuracy (A) = 0.441558
Harmonic Mean of F and A          = 0.285146


## Joint classifier experiment queue

In [8]:

# -------------------------
# Joint classifier experiment runner
# -------------------------
classifier_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)


def build_classifier_train_retrieval(claims_dict, retrieved_dict, source, input_top_k):
    if source not in {"retrieved", "gold", "gold_plus_retrieved"}:
        raise ValueError(source)
    out = {}
    for cid, claim in claims_dict.items():
        gold = [eid for eid in claim.get("evidences", []) if eid in evidence]
        retrieved = [eid for eid in retrieved_dict.get(cid, []) if eid in evidence]
        if source == "retrieved":
            chosen = retrieved
        elif source == "gold":
            chosen = gold
        else:
            chosen = []
            seen = set()
            for eid in gold + retrieved:
                if eid not in seen:
                    chosen.append(eid)
                    seen.add(eid)
                if len(chosen) >= input_top_k:
                    break
        if not chosen:
            chosen = retrieved[:1] or [next(iter(evidence.keys()))]
        out[cid] = chosen[:input_top_k]
    return out


class ClaimEvidenceDataset(Dataset):
    def __init__(self, claims_dict, evidence_dict, retrieval_dict, tokenizer, input_top_k, max_len):
        self.cids = list(claims_dict.keys())
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.retrieval = retrieval_dict
        self.tokenizer = tokenizer
        self.input_top_k = input_top_k
        self.max_len = max_len
        self.has_label = "claim_label" in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, idx):
        cid = self.cids[idx]
        claim = self.claims[cid]
        ev_ids = list(self.retrieval[cid])[: self.input_top_k]
        sep = f" {self.tokenizer.sep_token} "
        ev_texts = [self.evidence[eid] for eid in ev_ids if eid in self.evidence]
        if not ev_texts:
            ev_texts = [next(iter(self.evidence.values()))]
        ev_text = sep.join(ev_texts)
        enc = self.tokenizer(
            claim["claim_text"],
            ev_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.has_label:
            item["labels"] = torch.tensor(LABEL2ID[claim["claim_label"]], dtype=torch.long)
        return item


def make_classifier_loader(claims_dict, retrieval_dict, input_top_k, max_seq_len, batch_size, shuffle=False):
    validate_retrieval_coverage(claims_dict, retrieval_dict, split_name="classifier_loader")
    ds = ClaimEvidenceDataset(claims_dict, evidence, retrieval_dict, classifier_tokenizer, input_top_k, max_seq_len)
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def predict_labels(model, loader, dataset):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            logits = model(**batch).logits
            preds.extend(logits.argmax(dim=-1).detach().cpu().tolist())
    return {dataset.cids[i]: ID2LABEL[p] for i, p in enumerate(preds)}


def confusion_matrix_df(gold_claims, pred_labels):
    mat = pd.DataFrame(0, index=LABELS, columns=LABELS)
    for cid, claim in gold_claims.items():
        mat.loc[claim["claim_label"], pred_labels[cid]] += 1
    return mat


label_counts = Counter(c["claim_label"] for c in train_claims.values())
raw_class_weights = torch.tensor(
    [len(train_claims) / (len(LABELS) * label_counts[label]) for label in LABELS],
    dtype=torch.float32,
    device=DEVICE,
)
print("Raw class weights:", dict(zip(LABELS, raw_class_weights.detach().cpu().tolist())))


def class_weights_for_power(power):
    if power <= 0:
        return None
    weights = raw_class_weights.pow(float(power))
    weights = weights / weights.mean()
    return weights


def train_one_classifier_experiment(exp):
    set_seed(SEED)
    name = safe_name(exp["name"])
    input_top_k = int(exp["input_top_k"])
    max_seq_len = int(exp["max_seq_len"])
    weight_power = float(exp["class_weight_power"])
    schedule = list(exp["train_schedule"])
    schedule_name = "__".join(f"{src}{epochs}" for src, epochs in schedule)
    train_bs = min(TRAIN_BATCH_SIZE, 8) if max_seq_len >= 384 else TRAIN_BATCH_SIZE
    eval_bs = min(EVAL_BATCH_SIZE, 16) if max_seq_len >= 384 else EVAL_BATCH_SIZE
    ckpt_path = OUTPUT_DIR / f"classifier_best_{CACHE_TAG}_{name}.pt"

    print("\n" + "=" * 80)
    print("Classifier experiment:", exp)
    print("=" * 80)

    model = AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER_MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)

    dev_ds, dev_loader = make_classifier_loader(dev_claims, dev_retrieval, input_top_k, max_seq_len, eval_bs, shuffle=False)

    best_metrics = {"F": retrieval_majority_metrics["F"], "A": 0.0, "H": -1.0}
    best_state = None
    best_pred_labels = None
    pred_summary = ""

    if SAVE_MODEL_CHECKPOINTS and ckpt_path.exists() and not FORCE_CLASSIFIER_RETRAIN:
        print("Loading cached classifier checkpoint:", ckpt_path)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        best_metrics = ckpt.get("best_metrics", best_metrics)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=CLASSIFIER_LR, weight_decay=WEIGHT_DECAY)
        total_epochs = sum(int(e) for _, e in schedule)
        # Scheduler step count uses the first stage length as an approximation; exact enough for controlled comparison.
        example_retrieval = build_classifier_train_retrieval(train_claims, train_retrieval, schedule[0][0], input_top_k)
        _, example_loader = make_classifier_loader(train_claims, example_retrieval, input_top_k, max_seq_len, train_bs, shuffle=True)
        total_steps = max(1, len(example_loader) * total_epochs)
        scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_for_power(weight_power))

        global_epoch = 0
        for source, epochs in schedule:
            train_retrieval_for_stage = build_classifier_train_retrieval(train_claims, train_retrieval, source, input_top_k)
            _, train_loader = make_classifier_loader(train_claims, train_retrieval_for_stage, input_top_k, max_seq_len, train_bs, shuffle=True)
            print(f"Stage source={source}, epochs={epochs}, avg evidence={np.mean([len(v) for v in train_retrieval_for_stage.values()]):.2f}")
            for _ in range(int(epochs)):
                global_epoch += 1
                model.train()
                losses = []
                t0 = time.time()
                for batch in tqdm(train_loader, desc=f"{name} epoch {global_epoch}/{total_epochs}"):
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    labels = batch.pop("labels")
                    logits = model(**batch).logits
                    loss = loss_fn(logits, labels)
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    losses.append(float(loss.item()))

                pred_labels = predict_labels(model, dev_loader, dev_ds)
                predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=pred_labels)
                metrics = evaluate_submission(predictions, dev_claims, verbose=False)
                pred_counts = Counter(pred_labels.values())
                pred_summary = ", ".join(f"{label}:{pred_counts.get(label, 0)}" for label in LABELS)
                print(
                    f"Epoch {global_epoch}: loss={np.mean(losses):.4f} | "
                    f"F={metrics['F']:.4f} A={metrics['A']:.4f} H={metrics['H']:.4f} | preds=({pred_summary}) | "
                    f"time={time.time() - t0:.1f}s"
                )
                if metrics["H"] > best_metrics["H"]:
                    best_metrics = metrics
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    best_pred_labels = pred_labels

        if best_state is not None:
            model.load_state_dict(best_state)
        if SAVE_MODEL_CHECKPOINTS:
            torch.save({"model_state": model.state_dict(), "best_metrics": best_metrics, "experiment": exp}, ckpt_path)
            print("Saved classifier checkpoint:", ckpt_path)

    # Re-evaluate loaded/best model for robust row reporting.
    pred_labels = predict_labels(model, dev_loader, dev_ds)
    predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=pred_labels)
    metrics = evaluate_submission(predictions, dev_claims, verbose=False)
    cm = confusion_matrix_df(dev_claims, pred_labels)
    pred_counts = Counter(pred_labels.values())
    pred_summary = ", ".join(f"{label}:{pred_counts.get(label, 0)}" for label in LABELS)

    row = {
        "name": exp["name"],
        "input_top_k": input_top_k,
        "max_seq_len": max_seq_len,
        "class_weight_power": weight_power,
        "schedule": schedule_name,
        "dev_F": metrics["F"],
        "dev_A": metrics["A"],
        "dev_H": metrics["H"],
        "pred_summary": pred_summary,
        "SUPPORTS_acc": cm.loc["SUPPORTS", "SUPPORTS"] / max(1, cm.loc["SUPPORTS"].sum()),
        "REFUTES_acc": cm.loc["REFUTES", "REFUTES"] / max(1, cm.loc["REFUTES"].sum()),
        "NEI_acc": cm.loc["NOT_ENOUGH_INFO", "NOT_ENOUGH_INFO"] / max(1, cm.loc["NOT_ENOUGH_INFO"].sum()),
        "DISPUTED_acc": cm.loc["DISPUTED", "DISPUTED"] / max(1, cm.loc["DISPUTED"].sum()),
        "ckpt_path": str(ckpt_path),
    }

    del model
    torch.cuda.empty_cache()
    gc.collect()
    return row


Raw class weights: {'SUPPORTS': 0.5915221571922302, 'REFUTES': 1.5427135229110718, 'NOT_ENOUGH_INFO': 0.7953367829322815, 'DISPUTED': 2.475806474685669}


In [9]:

# Run all classifier experiments once, save a ranking table, and pick the best strategy.
classifier_rows = []
results_path = OUTPUT_DIR / "classifier_branch_experiment_results.csv"

for exp in CLASSIFIER_EXPERIMENTS:
    row = train_one_classifier_experiment(exp)
    classifier_rows.append(row)
    current = pd.DataFrame(classifier_rows).sort_values("dev_H", ascending=False).reset_index(drop=True)
    current.to_csv(results_path, index=False)
    display(current)
    print("Saved:", results_path)

classifier_results = pd.DataFrame(classifier_rows).sort_values("dev_H", ascending=False).reset_index(drop=True)
display(classifier_results)
print("Best classifier experiment:")
best_classifier_row = classifier_results.iloc[0].to_dict()
print(best_classifier_row)

majority_h = retrieval_majority_metrics["H"]
classifier_h = best_classifier_row["dev_H"]
use_majority_labels_for_final = (
    FALLBACK_TO_MAJORITY_IF_CLASSIFIER_WORSE and classifier_h < majority_h + CLASSIFIER_MIN_H_IMPROVEMENT
)
print(
    "Final label source:",
    "majority" if use_majority_labels_for_final else "classifier",
    f"(best classifier H={classifier_h:.4f}, majority H={majority_h:.4f}, required H>{majority_h + CLASSIFIER_MIN_H_IMPROVEMENT:.4f})",
)



Classifier experiment: {'name': 'v3_baseline_top5_len256_gpr_w0', 'input_top_k': 5, 'max_seq_len': 256, 'class_weight_power': 0.0, 'train_schedule': [('gold_plus_retrieved', 8)]}


[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4952.29it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Stage source=gold_plus_retrieved, epochs=8, avg evidence=4.77


v3_baseline_top5_len256_gpr_w0 epoch 1/8: 100%|██████████| 77/77 [00:06<00:00, 11.00it/s]


Epoch 1: loss=1.3041 | F=0.2106 A=0.4481 H=0.2865 | preds=(SUPPORTS:152, REFUTES:0, NOT_ENOUGH_INFO:2, DISPUTED:0) | time=7.3s


...

Epoch 8: loss=0.9147 | F=0.2106 A=0.4870 H=0.2940 | preds=(SUPPORTS:128, REFUTES:0, NOT_ENOUGH_INFO:26, DISPUTED:0) | time=12.4s
Saved classifier checkpoint: outputs_notebook_classifier_branch\classifier_best_classifier_branch_v5_from_v3_top500_joint_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_top4_len384_gold2_gpr6_w0.pt


,name,input_top_k,max_seq_len,class_weight_power,schedule,dev_F,dev_A,dev_H,pred_summary,SUPPORTS_acc,REFUTES_acc,NEI_acc,DISPUTED_acc,ckpt_path
0,top4_len384_gold2_retrieved6_w0,4,384,0.00,gold2__retrieved6,0.21056,0.500000,0.296329,"SUPPORTS:107, REFUTES:0, NOT_ENOUGH_INFO:47, D...",0.838235,0.0,0.487805,0.0,outputs_notebook_classifier_branch\classifier_...
1,top3_len384_gold2_retrieved6_w0,3,384,0.00,gold2__retrieved6,0.21056,0.500000,0.296329,"SUPPORTS:97, REFUTES:0, NOT_ENOUGH_INFO:57, DI...",0.779412,0.0,0.585366,0.0,outputs_notebook_classifier_branch\classifier_...
2,top4_len384_gold2_gpr6_w0,4,384,0.00,gold2__gold_plus_retrieved6,0.21056,0.493506,0.295178,"SUPPORTS:125, REFUTES:0, NOT_ENOUGH_INFO:29, D...",0.926471,0.0,0.317073,0.0,outputs_notebook_classifier_branch\classifier_...
3,top2_len384_gpr_w0,2,384,0.00,gold_plus_retrieved8,0.21056,0.487013,0.294006,"SUPPORTS:131, REFUTES:0, NOT_ENOUGH_INFO:23, D...",0.941176,0.0,0.268293,0.0,outputs_notebook_classifier_branch\classifier_...
4,v3_baseline_top5_len256_gpr_w0,5,256,0.00,gold_plus_retrieved8,0.21056,0.487013,0.294006,"SUPPORTS:121, REFUTES:0, NOT_ENOUGH_INFO:33, D...",0.882353,0.0,0.365854,0.0,outputs_notebook_classifier_branch\classifier_...
5,top3_len384_gold2_gpr6_w0,3,384,0.00,gold2__gold_plus_retrieved6,0.21056,0.487013,0.294006,"SUPPORTS:120, REFUTES:0, NOT_ENOUGH_INFO:34, D...",0.882353,0.0,0.365854,0.0,outputs_notebook_classifier_branch\classifier_...
6,top3_len384_gpr_w05,3,384,0.50,gold_plus_retrieved8,0.21056,0.480519,0.292812,"SUPPORTS:128, REFUTES:0, NOT_ENOUGH_INFO:26, D...",0.897059,0.0,0.317073,0.0,outputs_notebook_classifier_branch\classifier_...
7,top5_len384_gpr_w0,5,384,0.00,gold_plus_retrieved8,0.21056,0.480519,0.292812,"SUPPORTS:131, REFUTES:0, NOT_ENOUGH_INFO:23, D...",0.926471,0.0,0.268293,0.0,outputs_notebook_classifier_branch\classifier_...
8,top4_len384_gpr_w05,4,384,0.50,gold_plus_retrieved8,0.21056,0.480519,0.292812,"SUPPORTS:132, REFUTES:0, NOT_ENOUGH_INFO:22, D...",0.911765,0.0,0.292683,0.0,outputs_notebook_classifier_branch\classifier_...
9,top3_len384_gpr_w025,3,384,0.25,gold_plus_retrieved8,0.21056,0.474026,0.291595,"SUPPORTS:127, REFUTES:0, NOT_ENOUGH_INFO:27, D...",0.897059,0.0,0.292683,0.0,outputs_notebook_classifier_branch\classifier_...


Saved: outputs_notebook_classifier_branch\classifier_branch_experiment_results.csv


,name,input_top_k,max_seq_len,class_weight_power,schedule,dev_F,dev_A,dev_H,pred_summary,SUPPORTS_acc,REFUTES_acc,NEI_acc,DISPUTED_acc,ckpt_path
0,top4_len384_gold2_retrieved6_w0,4,384,0.00,gold2__retrieved6,0.21056,0.500000,0.296329,"SUPPORTS:107, REFUTES:0, NOT_ENOUGH_INFO:47, D...",0.838235,0.0,0.487805,0.0,outputs_notebook_classifier_branch\classifier_...
1,top3_len384_gold2_retrieved6_w0,3,384,0.00,gold2__retrieved6,0.21056,0.500000,0.296329,"SUPPORTS:97, REFUTES:0, NOT_ENOUGH_INFO:57, DI...",0.779412,0.0,0.585366,0.0,outputs_notebook_classifier_branch\classifier_...
2,top4_len384_gold2_gpr6_w0,4,384,0.00,gold2__gold_plus_retrieved6,0.21056,0.493506,0.295178,"SUPPORTS:125, REFUTES:0, NOT_ENOUGH_INFO:29, D...",0.926471,0.0,0.317073,0.0,outputs_notebook_classifier_branch\classifier_...
3,top2_len384_gpr_w0,2,384,0.00,gold_plus_retrieved8,0.21056,0.487013,0.294006,"SUPPORTS:131, REFUTES:0, NOT_ENOUGH_INFO:23, D...",0.941176,0.0,0.268293,0.0,outputs_notebook_classifier_branch\classifier_...
4,v3_baseline_top5_len256_gpr_w0,5,256,0.00,gold_plus_retrieved8,0.21056,0.487013,0.294006,"SUPPORTS:121, REFUTES:0, NOT_ENOUGH_INFO:33, D...",0.882353,0.0,0.365854,0.0,outputs_notebook_classifier_branch\classifier_...
5,top3_len384_gold2_gpr6_w0,3,384,0.00,gold2__gold_plus_retrieved6,0.21056,0.487013,0.294006,"SUPPORTS:120, REFUTES:0, NOT_ENOUGH_INFO:34, D...",0.882353,0.0,0.365854,0.0,outputs_notebook_classifier_branch\classifier_...
6,top3_len384_gpr_w05,3,384,0.50,gold_plus_retrieved8,0.21056,0.480519,0.292812,"SUPPORTS:128, REFUTES:0, NOT_ENOUGH_INFO:26, D...",0.897059,0.0,0.317073,0.0,outputs_notebook_classifier_branch\classifier_...
7,top5_len384_gpr_w0,5,384,0.00,gold_plus_retrieved8,0.21056,0.480519,0.292812,"SUPPORTS:131, REFUTES:0, NOT_ENOUGH_INFO:23, D...",0.926471,0.0,0.268293,0.0,outputs_notebook_classifier_branch\classifier_...
8,top4_len384_gpr_w05,4,384,0.50,gold_plus_retrieved8,0.21056,0.480519,0.292812,"SUPPORTS:132, REFUTES:0, NOT_ENOUGH_INFO:22, D...",0.911765,0.0,0.292683,0.0,outputs_notebook_classifier_branch\classifier_...
9,top3_len384_gpr_w025,3,384,0.25,gold_plus_retrieved8,0.21056,0.474026,0.291595,"SUPPORTS:127, REFUTES:0, NOT_ENOUGH_INFO:27, D...",0.897059,0.0,0.292683,0.0,outputs_notebook_classifier_branch\classifier_...


Best classifier experiment:
{'name': 'top4_len384_gold2_retrieved6_w0', 'input_top_k': 4, 'max_seq_len': 384, 'class_weight_power': 0.0, 'schedule': 'gold2__retrieved6', 'dev_F': 0.21055967841682127, 'dev_A': 0.5, 'dev_H': 0.2963293370177767, 'pred_summary': 'SUPPORTS:107, REFUTES:0, NOT_ENOUGH_INFO:47, DISPUTED:0', 'SUPPORTS_acc': 0.8382352941176471, 'REFUTES_acc': 0.0, 'NEI_acc': 0.4878048780487805, 'DISPUTED_acc': 0.0, 'ckpt_path': 'outputs_notebook_classifier_branch\\classifier_best_classifier_branch_v5_from_v3_top500_joint_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_top4_len384_gold2_retrieved6_w0.pt'}
Final label source: classifier (best classifier H=0.2963, majority H=0.2851, required H>0.2901)


## Export best classifier and test output

In [10]:

# Load the best classifier and write dev/test outputs.
best_exp_name = best_classifier_row["name"]
best_exp = next(e for e in CLASSIFIER_EXPERIMENTS if e["name"] == best_exp_name)
classifier_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)

best_classifier_model = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(DEVICE)
ckpt = torch.load(best_classifier_row["ckpt_path"], map_location=DEVICE)
best_classifier_model.load_state_dict(ckpt["model_state"])

input_top_k = int(best_exp["input_top_k"])
max_seq_len = int(best_exp["max_seq_len"])

def predict_for_claims_with_best(claims_dict, retrieval_dict):
    eval_bs = min(EVAL_BATCH_SIZE, 16) if max_seq_len >= 384 else EVAL_BATCH_SIZE
    ds, loader = make_classifier_loader(claims_dict, retrieval_dict, input_top_k, max_seq_len, eval_bs, shuffle=False)
    return predict_labels(best_classifier_model, loader, ds)

if use_majority_labels_for_final:
    final_dev_pred_labels = {cid: majority for cid in dev_claims.keys()}
else:
    final_dev_pred_labels = predict_for_claims_with_best(dev_claims, dev_retrieval)

dev_predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=final_dev_pred_labels)
print("Selected final dev score:")
final_dev_metrics = evaluate_submission(dev_predictions, dev_claims)
write_predictions(dev_predictions, OUTPUT_DIR / "dev-predictions.json")

print("Confusion matrix for selected final labels:")
display(confusion_matrix_df(dev_claims, final_dev_pred_labels))

# Test prediction generation. The test set is unlabeled; do not inspect or manually modify predictions.
test_bm25_candidates = compute_or_load_bm25_candidates(test_claims, "test", BM25_CANDIDATE_K)
test_ce_scores = score_candidates_with_reranker(reranker_model, test_claims, test_bm25_candidates, "test_final", allow_cache=True)
test_retrieval = apply_retrieval_setting(test_ce_scores, best_row)
validate_retrieval_coverage(test_claims, test_retrieval, split_name="test")

if use_majority_labels_for_final:
    test_pred_labels = {cid: majority for cid in test_claims.keys()}
else:
    test_pred_labels = predict_for_claims_with_best(test_claims, test_retrieval)

test_predictions = build_predictions(test_claims, test_retrieval, label_predictions=test_pred_labels)
write_predictions(test_predictions, OUTPUT_DIR / "test-output.json")

config_path = OUTPUT_DIR / "best_classifier_branch_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "best_classifier": best_classifier_row,
        "retrieval_setting": best_row,
        "retrieval_majority_metrics": retrieval_majority_metrics,
        "final_dev_metrics": final_dev_metrics,
        "final_label_source": "majority" if use_majority_labels_for_final else "classifier",
    }, f, indent=2, ensure_ascii=True)
print("Saved:", config_path)
print("Test output ready:", OUTPUT_DIR / "test-output.json")

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6352.99it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Selected final dev score:
Evidence Retrieval F-score (F)    = 0.210560
Claim Classification Accuracy (A) = 0.500000
Harmonic Mean of F and A          = 0.296329
Wrote outputs_notebook_classifier_branch\dev-predictions.json
Confusion matrix for selected final labels:


,SUPPORTS,REFUTES,NOT_ENOUGH_INFO,DISPUTED
SUPPORTS,57,0,11,0
REFUTES,15,0,12,0
NOT_ENOUGH_INFO,21,0,20,0
DISPUTED,14,0,4,0


Computing BM25 candidates for test ...


Scoring test_final candidates with cross-encoder reranker...


100%|██████████| 153/153 [00:47<00:00,  3.25it/s]


Cached CE scores: outputs_notebook_classifier_branch\cache\test_final_classifier_branch_v5_from_v3_top500_joint_cross-encoder-ms-marco-MiniLM-L-6-v2_top500_seed42_ce_scores.pkl
Wrote outputs_notebook_classifier_branch\test-output.json
Saved: outputs_notebook_classifier_branch\best_classifier_branch_config.json
Test output ready: outputs_notebook_classifier_branch\test-output.json
